# Linear Supervised-Learning Models

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Loading the split dataset
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

features_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_model_features.parquet'
)

baseline_metrics_path = (
    project_root
    / 'reports'
    / 'baseline_validation_metrics.csv'
)

feature_df = pd.read_parquet(
    features_path
)

feature_df['date'] = pd.to_datetime(
    feature_df['date']
)

feature_df = feature_df.sort_values(
    ['store_id', 'item_id', 'date']
).reset_index(drop=True)

print(f'Rows: {len(feature_df):,}')
print(f'Items: {feature_df["item_id"].nunique():,}')
print(
    f'Date range: '
    f'{feature_df["date"].min().date()} '
    f'to {feature_df["date"].max().date()}'
)

Rows: 166,886
Items: 108
Date range: 2011-02-26 to 2016-05-22


In [3]:
all_dates = pd.DatetimeIndex(
    sorted(feature_df['date'].unique())
)

train_dates = all_dates[:-56]
validation_dates = all_dates[-56:-28]
test_dates = all_dates[-28:]

training_df = feature_df.loc[feature_df['date'].isin(train_dates)].copy()

validation_df = feature_df.loc[feature_df['date'].isin(validation_dates)].copy()

In [4]:
# Date checks
print(
    f'Training: {train_dates.min().date()} '
    f'to {train_dates.max().date()}'
)

print(
    f'Validation: {validation_dates.min().date()} '
    f'to {validation_dates.max().date()}'
)

print(
    f'Testing: {test_dates.min().date()} '
    f'to {test_dates.max().date()}'
)

print(f'Training rows: {len(training_df):,}')
print(f'Validation rows: {len(validation_df):,}')

assert train_dates.max() < validation_dates.min()
assert validation_dates.max() < test_dates.min()

Training: 2011-02-26 to 2016-03-27
Validation: 2016-03-28 to 2016-04-24
Testing: 2016-04-25 to 2016-05-22
Training rows: 160,838
Validation rows: 3,024


### Selecting the provisional predictors

In [5]:
numeric_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'sell_price',
    'price_change',
    'event_indicator'
]

In [6]:
categorical_features = [
    'weekday',
    'month',
    'item_id',
    'dept_id',
    'cat_id'
]

model_features = (
    numeric_features
    + categorical_features
)

# Sales as target feature
target = 'sales'

assert target not in model_features

In [7]:
# Checking any missing training values
missing_training_values = (
    training_df[model_features + [target]]
    .isna()
    .sum()
)

missing_training_values[missing_training_values > 0]

Series([], dtype: int64)

Empty series confirmed the check

### Preprocessing pipeline

In [8]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features)
    ],
    sparse_threshold=1.0
)

### Linear and Ridge regressions definition

In [10]:
models = {
    'linear_regression': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('regressor', LinearRegression())]),

    'ridge_regression': Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('regressor',Ridge(
                alpha=1.0,  # Initial hyperparameter value
                solver='lsqr'
            )
        )
    ])
}

### Training on both models

In [11]:
X_train = training_df[model_features]

y_train = training_df[target]

In [12]:
trained_models = {}

for model_name, model in models.items():
    print(f'Training {model_name}...')

    model.fit(X_train, y_train)

    trained_models[model_name] = model

    print(f'Finished {model_name}.')

Training linear_regression...
Finished linear_regression.
Training ridge_regression...
Finished ridge_regression.


### Recursive ML Forecast

In [13]:
series_keys = [
    'store_id',
    'item_id'
]

In [15]:
# recursive function
def recursive_ml_forecast(
    model,
    data,
    forecast_dates,
    feature_columns,
    model_name
):
    forecast_dates = pd.DatetimeIndex(
        forecast_dates
    )

    forecast_start = forecast_dates.min()

    histories = {}

    for series_values, group in data.groupby(
        series_keys,
        sort=False
    ):
        history = (
            group.loc[
                group['date'] < forecast_start,
                'sales'
            ]
            .astype(float)
            .tolist()
        )

        if len(history) < 28:
            raise ValueError(
                f'{series_values} has fewer than '
                f'28 historical observations.'
            )

        histories[series_values] = history

    prediction_frames = []

    for forecast_date in forecast_dates:
        day_rows = data.loc[
            data['date'] == forecast_date
        ].sort_values(
            series_keys
        ).copy()

        for row_index, row in day_rows.iterrows():
            series_values = tuple(
                row[column]
                for column in series_keys
            )

            history = histories[series_values]

            day_rows.at[row_index, 'lag_1'] = history[-1]

            day_rows.at[row_index, 'lag_7'] = history[-7]

            day_rows.at[row_index, 'lag_14'] = history[-14]

            day_rows.at[row_index, 'lag_28'] = history[-28]

            day_rows.at[row_index, 'rolling_mean_7'] = np.mean(history[-7:])

            day_rows.at[row_index, 'rolling_mean_28'] = np.mean(history[-28:])

            day_rows.at[row_index, 'rolling_std_7'] = np.std(history[-7:],ddof=1)

        assert not day_rows[feature_columns].isna().any().any()

        raw_forecasts = model.predict(day_rows[feature_columns])

        # clipping top 0
        clipped_forecasts = np.maximum(raw_forecasts, 0)

        output = day_rows[
            [
                'store_id',
                'item_id',
                'dept_id',
                'cat_id',
                'date',
                'sales'
            ]
        ].copy()

        output['model'] = model_name
        output['raw_forecast'] = raw_forecasts
        output['forecast'] = clipped_forecasts

        prediction_frames.append(output)

        # Use clipped predictions to construct
        # subsequent forecast-day features.
        for row, prediction in zip(
            day_rows.itertuples(index=False),
            clipped_forecasts
        ):
            series_values = (
                row.store_id,
                row.item_id
            )

            histories[series_values].append(float(prediction))

    return pd.concat(
        prediction_frames,
        ignore_index=True
    )

### Validation set predictions

In [16]:
prediction_frames = []

for model_name, model in trained_models.items():
    print(f'Forecasting with {model_name}...')

    model_predictions = (
        recursive_ml_forecast(
            model=model,
            data=feature_df,
            forecast_dates=validation_dates,
            feature_columns=model_features,
            model_name=model_name
        )
    )

    prediction_frames.append(model_predictions)

validation_ml_predictions = pd.concat(
    prediction_frames,
    ignore_index=True
)

validation_ml_predictions.head()

Forecasting with linear_regression...
Forecasting with ridge_regression...


,store_id,item_id,dept_id,cat_id,date,sales,model,raw_forecast,forecast
0,CA_1,FOODS_1_010,FOODS_1,FOODS,2016-03-28,0,linear_regression,0.055832,0.055832
1,CA_1,FOODS_2_016,FOODS_2,FOODS,2016-03-28,2,linear_regression,3.320069,3.320069
2,CA_1,FOODS_2_071,FOODS_2,FOODS,2016-03-28,3,linear_regression,0.423753,0.423753
3,CA_1,FOODS_2_149,FOODS_2,FOODS,2016-03-28,3,linear_regression,0.873647,0.873647
4,CA_1,FOODS_2_246,FOODS_2,FOODS,2016-03-28,0,linear_regression,0.060926,0.060926


In [17]:
# Checking the size
expected_prediction_rows = (
    108
    * 28
    * len(trained_models)
)

print(
    f'Prediction rows: '
    f'{len(validation_ml_predictions):,}'
)

print(
    f'Expected rows: '
    f'{expected_prediction_rows:,}'
)

assert (len(validation_ml_predictions) == expected_prediction_rows
)

Prediction rows: 6,048
Expected rows: 6,048


In [18]:
# Investigating the negative prediction
negative_summary = (
    validation_ml_predictions
    .assign(
        negative_prediction=lambda data:
        data['raw_forecast'] < 0
    )
    .groupby('model')
    .agg(
        total_predictions=(
            'raw_forecast',
            'size'
        ),
        negative_predictions=(
            'negative_prediction',
            'sum'
        ),
        minimum_prediction=(
            'raw_forecast',
            'min'
        )
    )
    .reset_index()
)

negative_summary[
    'negative_percentage'
] = (
    negative_summary[
        'negative_predictions'
    ]
    / negative_summary[
        'total_predictions'
    ]
    * 100
)

negative_summary

,model,total_predictions,negative_predictions,minimum_prediction,negative_percentage
0,linear_regression,3024,123,-0.169724,4.067460
1,ridge_regression,3024,109,-0.148041,3.604497


Negative predictions are expected to be possible because Linear and Ridge Regression do not constrain outputs to non-negative values.

In [19]:
# Function to calculate raw and clipped metrics
def calculate_prediction_metrics(
    predictions,
    prediction_column
):
    actual = predictions[
        'sales'
    ].to_numpy(dtype=float)

    forecast = predictions[
        prediction_column
    ].to_numpy(dtype=float)

    errors = actual - forecast

    mae = np.mean(
        np.abs(errors)
    )

    rmse = np.sqrt(
        np.mean(errors ** 2)
    )

    denominator = np.sum(
        np.abs(actual)
    )

    wape = (
        np.sum(np.abs(errors))
        / denominator
        if denominator > 0
        else np.nan
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'WAPE': wape
    }

In [20]:
# Loop to get the score on all validation metrics
metric_rows = []

for model_name, group in (
    validation_ml_predictions.groupby('model')
):
    raw_metrics = (
        calculate_prediction_metrics(
            group,
            'raw_forecast'
        )
    )

    clipped_metrics = (
        calculate_prediction_metrics(
            group,
            'forecast'
        )
    )

    metric_rows.append({
        'model': model_name,
        'prediction_version': 'raw',
        **raw_metrics
    })

    metric_rows.append({
        'model': model_name,
        'prediction_version': 'clipped',
        **clipped_metrics
    })

ml_validation_metrics = (
    pd.DataFrame(metric_rows)
    .sort_values(
        ['prediction_version', 'WAPE']
    )
    .reset_index(drop=True)
)

ml_validation_metrics

,model,prediction_version,MAE,RMSE,WAPE
0,linear_regression,clipped,1.323802,2.459765,0.844908
1,ridge_regression,clipped,1.328547,2.462850,0.847937
2,linear_regression,raw,1.326096,2.459855,0.846373
3,ridge_regression,raw,1.330277,2.462904,0.849042


In [21]:
# Compare against the baselines
baseline_metrics = pd.read_csv(baseline_metrics_path)

baseline_metrics

,model,MAE,RMSE,WAPE
0,rolling_mean_28,1.169400,2.417961,0.746363
1,seasonal_naive_7,1.432540,3.151173,0.914310
2,previous_day,1.507937,3.461905,0.962431


In [22]:
# Combining all metrics for comparison
clipped_ml_metrics = (
    ml_validation_metrics.loc[
        ml_validation_metrics['prediction_version'] == 'clipped',
        [
            'model',
            'MAE',
            'RMSE',
            'WAPE'
        ]
    ]
)

model_comparison = pd.concat(
    [
        baseline_metrics,
        clipped_ml_metrics
    ],
    ignore_index=True
).sort_values(
    'WAPE'
).reset_index(drop=True)

model_comparison[
    'WAPE_percentage'
] = (model_comparison['WAPE']* 100)

model_comparison

,model,MAE,RMSE,WAPE,WAPE_percentage
0,rolling_mean_28,1.169400,2.417961,0.746363,74.636250
1,linear_regression,1.323802,2.459765,0.844908,84.490836
2,ridge_regression,1.328547,2.462850,0.847937,84.793721
3,seasonal_naive_7,1.432540,3.151173,0.914310,91.430984
4,previous_day,1.507937,3.461905,0.962431,96.243141


### Inspecting the engineered features coeffs.

In [24]:
coefficient_frames = []

for model_name, model in (
    trained_models.items()
):
    feature_names = (
        model.named_steps['preprocessor'].get_feature_names_out()
        )

    coefficients = (
        model.named_steps['regressor'].coef_
    )

    coefficient_frame = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefficients
    })

    coefficient_frame[
        'absolute_coefficient'
    ] = coefficient_frame[
        'coefficient'
    ].abs()

    coefficient_frame['model'] = model_name

    coefficient_frames.append(coefficient_frame)

coefficient_df = pd.concat(
    coefficient_frames,
    ignore_index=True
)

In [25]:
# Numerical features
numeric_coefficients = (
    coefficient_df.loc[
        coefficient_df[
            'feature'
        ].str.startswith('numeric__')
    ]
    .assign(
        feature=lambda data:
        data['feature'].str.replace(
            'numeric__',
            '',
            regex=False
        )
    )
    .sort_values(
        [
            'model',
            'absolute_coefficient'
        ],
        ascending=[
            True,
            False
        ]
    )
)

numeric_coefficients

,feature,coefficient,absolute_coefficient,model
4,rolling_mean_7,1.348565,1.348565,linear_regression
6,rolling_std_7,-0.422528,0.422528,linear_regression
5,rolling_mean_28,0.413650,0.413650,linear_regression
7,sell_price,-0.393895,0.393895,linear_regression
0,lag_1,0.353909,0.353909,linear_regression
2,lag_14,0.310806,0.310806,linear_regression
3,lag_28,0.236035,0.236035,linear_regression
1,lag_7,0.141110,0.141110,linear_regression
9,event_indicator,-0.053131,0.053131,linear_regression
8,price_change,-0.000052,0.000052,linear_regression


In [26]:
# Saving
models_directory = (
    project_root
    / 'models'
)

reports_directory = (
    project_root
    / 'reports'
)

predictions_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_validation_linear_predictions.parquet'
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

reports_directory.mkdir(
    parents=True,
    exist_ok=True
)

predictions_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
# Saving models in joblib
for model_name, model in trained_models.items():
    model_path = (
        models_directory
        / f'{model_name}_pipeline.joblib'
    )

    joblib.dump(
        model,
        model_path
    )

    print(f'Saved model: {model_path}')

validation_ml_predictions.to_parquet(
    predictions_path,
    index=False
)

ml_validation_metrics.to_csv(
    reports_directory
    / 'linear_model_validation_metrics.csv',
    index=False
)

model_comparison.to_csv(
    reports_directory
    / 'validation_model_comparison.csv',
    index=False
)

numeric_coefficients.to_csv(
    reports_directory
    / 'linear_model_numeric_coefficients.csv',
    index=False
)

negative_summary.to_csv(
    reports_directory
    / 'linear_model_negative_predictions.csv',
    index=False
)

Saved model: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\linear_regression_pipeline.joblib
Saved model: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\models\ridge_regression_pipeline.joblib
